1. Simulated Annealing (Luyện Kim)

In [1]:
import random
import math

# Cấu hình hướng di chuyển dùng chung (Thứ tự ưu tiên: L -> R -> U -> D)
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def run_simulated_annealing(start, goal, max_steps):
    current = start
    path = []
    states_history = []

    curr_h = manhattan_distance(current, goal)
    states_history.append((current, f"Bắt đầu Luyện Kim tại START. h = {curr_h}, Nhiệt độ ban đầu T = 100.0"))

    # Thiết lập các tham số tôi luyện kim
    T = 100.0          # Nhiệt độ ban đầu
    alpha = 0.95        # Hệ số hạ nhiệt (cooling rate)

    while current != goal:
        if len(states_history) >= max_steps:
            return None, states_history

        if T < 0.001:
            states_history.append((current, f"Lò nguội hoàn toàn (T đạt giới hạn dưới). Thuật toán dừng lại tại h = {curr_h}."))
            return None, states_history

        x, y = find_zero(current)
        valid_neighbors = []

        # Lấy tất cả các láng giềng hợp lệ quanh vị trí ô trống
        for move, (dx, dy), move_name in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                neighbor = swap(current, x, y, nx, ny)
                neighbor_h = manhattan_distance(neighbor, goal)
                valid_neighbors.append((neighbor, neighbor_h, move, move_name))

        if not valid_neighbors:
            return None, states_history

        # Chọn ngẫu nhiên MỘT láng giềng bất kỳ (bản chất của SA là chọn ngẫu nhiên trạng thái kế tiếp)
        next_state, next_h, move, move_name = random.choice(valid_neighbors)

        # Tính toán độ chênh lệch năng lượng (Delta E). Vì ta muốn giảm h nên Delta E = h_cũ - h_mới
        delta_e = curr_h - next_h

        if delta_e > 0:
            # Trạng thái mới tốt hơn -> Chấp nhận ngay lập tức
            current = next_state
            curr_h = next_h
            path.append(move)
            states_history.append((current, f"Hướng tốt: Di chuyển [{move}] {move_name}. h mới = {curr_h} (Giảm được {-delta_e} đơn vị). T = {T:.3f}"))
        else:
            # Trạng thái mới tệ hơn hoặc bằng -> Chấp nhận với xác suất Boltzmann P = e^(delta_e / T)
            prob = math.exp(delta_e / T)
            rand_val = random.random()

            if rand_val < prob:
                # Chấp nhận bước nhảy tệ này để phá kẹt
                current = next_state
                curr_h = next_h
                path.append(move)
                states_history.append((current, f"Chấp nhận hướng tệ hơn nhờ nhiệt độ! Hướng [{move}] {move_name}. h tăng lên {curr_h} (Xác suất P={prob:.3f} > {rand_val:.3f}). T = {T:.3f}"))
            else:
                # Từ chối, giữ nguyên trạng thái cũ ở bước này
                states_history.append((current, f"Từ chối hướng tệ [{move}] {move_name} (h={next_h}). Xác suất P={prob:.3f} <= {rand_val:.3f}. Giữ nguyên. T = {T:.3f}"))

        # Hạ nhiệt độ theo chu kỳ
        T *= alpha

    return path, states_history

if __name__ == "__main__":
    start = ((2, 8, 3), (1, 6, 4), (7, 0, 5))  # 0 là ô trống
    goal  = ((1, 2, 3), (8, 0, 4), (7, 6, 5))

    path, history = run_simulated_annealing(start, goal, max_steps=1000)
    if path:
        print(f"Thành công! Số bước đi: {len(path)}")
        print(f"Lộ trình: {' -> '.join(path)}")
    else:
        print("Chưa tìm ra đường đi trong giới hạn bước duyệt.")

Thành công! Số bước đi: 13
Lộ trình: U -> R -> L -> U -> L -> D -> U -> R -> D -> U -> L -> D -> R


2. Bidirectional Search (Tìm kiếm hai hướng)

In [2]:
from collections import deque

# Cấu hình hướng di chuyển dùng chung (Thứ tự ưu tiên: L -> R -> U -> D)
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def run_bidirectional_search(start, goal, max_steps):
    # Hai hàng đợi cho hai đầu
    queue_f = deque([(start, [])])  # Xuôi từ Start (Forward)
    queue_b = deque([(goal, [])])   # Ngược từ Goal (Backward)

    # Hai tập đã duyệt lưu vết cùng lộ trình tương ứng dẫn đến trạng thái đó
    visited_f = {start: []}
    visited_b = {goal: []}

    states_history = []
    states_history.append((start, "Khởi tạo Tìm kiếm Hai hướng: Nhánh Xuôi (Start) và Nhánh Ngược (Goal) bắt đầu quét song song."))

    # Bản đồ đảo ngược ký tự hướng di chuyển để tính toán lộ trình nhánh Ngược
    reverse_move = {'L': 'R', 'R': 'L', 'U': 'D', 'D': 'U'}

    step_count = 0
    while queue_f and queue_b:
        if len(states_history) >= max_steps:
            return None, states_history

        # 1. Phát triển 1 bước bên phía Nhánh Xuôi (Forward)
        curr_f, path_f = queue_f.popleft()
        states_history.append((curr_f, f"[Nhánh Xuôi] Đang xét một Node. Kích thước tập duyệt xuôi: {len(visited_f)}"))

        # Kiểm tra giao điểm ngay lập tức
        if curr_f in visited_b:
            # Hai nhánh đã chạm nhau! Kết hợp lộ trình
            full_path = path_f + visited_b[curr_f]
            states_history.append((curr_f, "HAI MA TRẬN ĐÃ GẶP NHAU TẠI ĐÂY! Hoàn thành kết nối lộ trình giữa Start và Goal."))
            return full_path, states_history

        x, y = find_zero(curr_f)
        for move, (dx, dy), move_name in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                next_f = swap(curr_f, x, y, nx, ny)
                if next_f not in visited_f:
                    visited_f[next_f] = path_f + [move]
                    queue_f.append((next_f, path_f + [move]))

        # 2. Phát triển 1 bước bên phía Nhánh Ngược (Backward)
        curr_b, path_b = queue_b.popleft()
        states_history.append((curr_b, f"[Nhánh Ngược] Đang xét một Node. Kích thước tập duyệt ngược: {len(visited_b)}"))

        # Kiểm tra giao điểm
        if curr_b in visited_f:
            # Hai nhánh gặp nhau
            full_path = visited_f[curr_b] + path_b
            states_history.append((curr_b, "HAI MA TRẬN ĐÃ GẶP NHAU TẠI ĐÂY! Hoàn thành kết nối lộ trình giữa Start và Goal."))
            return full_path, states_history

        x, y = find_zero(curr_b)
        for move, (dx, dy), move_name in ACTIONS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                next_b = swap(curr_b, x, y, nx, ny)
                if next_b not in visited_b:
                    # Hành động đẩy ô trống nhánh ngược từ Goal lên cần đảo ngược hướng di chuyển thực tế
                    actual_move = reverse_move[move]
                    # Đường đi nhánh ngược được chèn vào ĐẦU danh sách để đảm bảo đúng thứ tự khi nối chuỗi
                    new_path_b = [actual_move] + path_b
                    visited_b[next_b] = new_path_b
                    queue_b.append((next_b, new_path_b))

        step_count += 1

    return None, states_history

if __name__ == "__main__":
    start = ((2, 8, 3), (1, 6, 4), (7, 0, 5))  # 0 là ô trống
    goal  = ((1, 2, 3), (8, 0, 4), (7, 6, 5))

    path, history = run_bidirectional_search(start, goal, max_steps=2000)
    if path:
        print(f"Thành công! Số bước đi: {len(path)}")
        print(f"Lộ trình: {' -> '.join(path)}")
    else:
        print("Không tìm thấy giao điểm giữa hai ma trận.")

Thành công! Số bước đi: 5
Lộ trình: U -> U -> L -> D -> R
